# Bundle Migration: Generate & Deploy (Modular)

## Overview

Generate Databricks Asset Bundle and deploy to target workspace using configuration from databricks.yml.

**Prerequisites:**
- ✅ Bundle_03 completed (dashboards exported and transformed)
- ✅ databricks.yml configured with target workspace details

**What this notebook does:**
1. Loads transformed dashboards and permissions
2. Generates complete bundle structure
3. Validates bundle configuration
4. Deploys to target workspace via CLI
5. Verifies deployment

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml databricks-cli "numpy<2" --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Load Config

In [ ]:
import sys
import os
from pathlib import Path

# Dynamically locate helpers directory
print("=== HELPERS PATH RESOLUTION DEBUG ===")

try:
    # In Databricks workspace/job context
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print(f"Notebook path (from dbutils): {notebook_path}")
    
    # Add /Workspace prefix if not present
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
        print(f"Added /Workspace prefix: {notebook_path}")
    
    # Get parent directory of Bundle folder
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    print(f"Notebook dir: {notebook_dir}")
    print(f"Bundle parent: {bundle_parent}")
    
    # Verify helpers exists in bundle_parent
    helpers_path = os.path.join(bundle_parent, 'helpers')
    print(f"\nChecking helpers path: {helpers_path}")
    
    if os.path.exists(helpers_path):
        print(f"  Helpers directory EXISTS")
        init_file = os.path.join(helpers_path, '__init__.py')
        if os.path.exists(init_file):
            print(f"  __init__.py FOUND")
            print(f"  Contents: {os.listdir(helpers_path)[:10]}")
        else:
            print(f"  WARNING: No __init__.py found")
    else:
        print(f"  WARNING: Helpers directory does not exist at {helpers_path}")
    
    # Add BUNDLE PARENT to sys.path (not the helpers dir itself)
    sys_path_entry = bundle_parent
    
except Exception as e:
    print(f"Error in workspace context: {e}")
    import traceback
    traceback.print_exc()
    # Fallback for local execution
    sys_path_entry = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f"Using local fallback: {sys_path_entry}")

sys_path_entry = os.path.normpath(sys_path_entry)

# Add bundle parent to sys.path (NOT the helpers directory)
print(f"\nAdding to sys.path: {sys_path_entry}")
if sys_path_entry not in sys.path:
    sys.path.insert(0, sys_path_entry)
    print(f"  Added to sys.path")
else:
    print(f"  Already in sys.path")

print(f"\nsys.path first 3 entries: {sys.path[:3]}")
print("=== END DEBUG ===\n")

from helpers import (
    set_config,
    get_path,
    write_volume_file,
    ensure_directory_exists,
    create_workspace_client,
    list_volume_files,
    load_permissions_from_volume,
    generate_bundle_structure,
    validate_bundle,
    deploy_bundle
)
import pandas as pd

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
# ============================================================================
# CONFIGURATION FROM PARAMETERS (databricks.yml)
# ============================================================================
# This notebook reads configuration from job parameters (widgets)
# All configuration is passed from databricks.yml

print("="*80)
print("LOADING CONFIGURATION FROM PARAMETERS")
print("="*80)

# Read all parameters from widgets
catalog = dbutils.widgets.get('catalog')
volume_base = dbutils.widgets.get('volume_base')
source_workspace_url = dbutils.widgets.get('source_workspace_url')
target_workspace_url = dbutils.widgets.get('target_workspace_url')
exported_path_rel = dbutils.widgets.get('exported_path')
transformed_path_rel = dbutils.widgets.get('transformed_path')
bundles_path_rel = dbutils.widgets.get('bundles_path')
target_parent_path = dbutils.widgets.get('target_parent_path')
warehouse_name = dbutils.widgets.get('warehouse_name')
bundle_name = dbutils.widgets.get('bundle_name')
apply_permissions = dbutils.widgets.get('apply_permissions').lower() == 'true'
permissions_dry_run = dbutils.widgets.get('permissions_dry_run').lower() == 'true'
embed_credentials = dbutils.widgets.get('embed_credentials').lower() == 'true'

# Build full paths
exported_path = f"{volume_base}/{exported_path_rel}"
transformed_path = f"{volume_base}/{transformed_path_rel}"
bundles_path = f"{volume_base}/{bundles_path_rel}"

# Build config dict for helper functions
config = {
    'source': {
        'workspace_url': source_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'target': {
        'workspace_url': target_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'paths': {
        'volume_base': volume_base,
        'exported': exported_path_rel,
        'transformed': transformed_path_rel,
        'bundles': bundles_path_rel,
        'target_parent_path': target_parent_path
    },
    'bundle': {
        'name': bundle_name,
        'embed_credentials': embed_credentials
    },
    'warehouse': {
        'warehouse_name': warehouse_name
    },
    'permissions': {
        'apply': apply_permissions,
        'dry_run': permissions_dry_run
    }
}

# Cache config for helper functions
set_config(config)

print("\n✅ Configuration loaded from parameters\n")
print(f"   Source: {source_workspace_url}")
print(f"   Target: {target_workspace_url}")
print(f"   Volume base: {volume_base}")
print(f"   Bundle name: {bundle_name}")
print(f"   Parent path: {target_parent_path}")
print(f"   Warehouse: {warehouse_name}")
print(f"   Embed credentials: {embed_credentials}")

# Ensure directories exist
print("\n📁 Ensuring directories exist...")
ensure_directory_exists(bundles_path)
print(f"   ✅ Bundles: {bundles_path}")

## Cell 3: Load Transformed Dashboards & Permissions

In [ ]:
print("\n" + "="*80)
print("LOADING DASHBOARDS & PERMISSIONS")
print("="*80)

# Use paths already defined in config
print(f"\n📂 Loading transformed dashboards from: {transformed_path}")
dashboard_files = list_volume_files(transformed_path, '*.lvdash.json')

if not dashboard_files:
    print("   ❌ No transformed dashboards found!")
    print("   💡 Run Bundle_01 first to export and transform dashboards")
    raise Exception("No dashboards to deploy")

print(f"   ✅ Found {len(dashboard_files)} dashboard(s)")

# Load permissions
print(f"\n🔐 Loading permissions from: {exported_path}")
permissions_map = load_permissions_from_volume(exported_path)
print(f"   ✅ Loaded permissions for {len(permissions_map)} dashboard(s)")

# Build dashboard list with metadata
dashboards = []
for file_path in dashboard_files:
    filename = Path(file_path).name
    
    # Extract dashboard ID and name from filename
    # Format: dashboard_{id}_{name}.lvdash.json
    parts = filename.replace('.lvdash.json', '').split('_', 2)
    if len(parts) >= 3:
        dashboard_id = parts[1]
        name = parts[2].replace('_', ' ')
    else:
        dashboard_id = filename
        name = filename
    
    dashboards.append({
        'id': dashboard_id,
        'name': name,
        'json_path': file_path
    })

# Display summary
df = pd.DataFrame([{'Dashboard': d['name'], 'ID': d['id']} for d in dashboards])
display(df)

## Cell 4: Generate Bundle Structure

In [ ]:
print("\n" + "="*80)
print("GENERATING BUNDLE STRUCTURE")
print("="*80)

# Use bundles_path already defined
print(f"\n📦 Generating bundle: {bundle_name}")
print(f"   Output path: {bundles_path}")

try:
    bundle_path = generate_bundle_structure(
        bundle_name=bundle_name,
        target_workspace_url=target_workspace_url,
        transformed_dashboards=dashboards,
        permissions_map=permissions_map,
        bundle_output_path=bundles_path,
        warehouse_id=None,
        warehouse_name=warehouse_name,
        parent_path=target_parent_path,
        embed_credentials=embed_credentials
    )
    
    print(f"\n✅ Bundle generated at: {bundle_path}")
    print("\n📋 Bundle structure:")
    print("   ├── databricks.yml")
    print("   ├── resources/")
    print("   │   └── dashboards.yml")
    print("   └── src/")
    print("       └── dashboards/")
    print(f"           └── {len(dashboards)} .lvdash.json file(s)")
    
except Exception as e:
    print(f"\n❌ Failed to generate bundle: {e}")
    raise

## Cell 5: Validate Bundle

In [ ]:
print("\n" + "="*80)
print("VALIDATING BUNDLE")
print("="*80)

print("\n🔍 Running: databricks bundle validate")

# In serverless compute, volume paths are directly accessible
# No /dbfs prefix needed - use volume path as-is
result = validate_bundle(bundle_path)

if result['success']:
    print("\n✅ Bundle validation passed")
    if result.get('stdout'):
        print(f"\n{result['stdout']}")
else:
    print("\n❌ Bundle validation failed")
    if result.get('stderr'):
        print(f"\nError: {result['stderr']}")
    if result.get('error'):
        print(f"\nError: {result['error']}")
    raise Exception("Bundle validation failed")

## Cell 6: Deploy Bundle

In [ ]:
print("\n" + "="*80)
print("DEPLOYING BUNDLE")
print("="*80)

print(f"\n🚀 Deploying to: {target_workspace_url}")
print("   This may take a few minutes...")

deploy_result = deploy_bundle(bundle_path)

if deploy_result['success']:
    print("\n✅ Bundle deployed successfully!")
    
    if deploy_result.get('stdout'):
        print(f"\n{deploy_result['stdout']}")
    
    print("\n" + "="*80)
    print("DEPLOYMENT SUMMARY")
    print("="*80)
    print(f"\n📊 Dashboards deployed: {len(dashboards)}")
    print(f"📁 Location: {target_parent_path}")
    print(f"🔗 Workspace: {target_workspace_url}")
    
    print("\n✅ Next steps:")
    print("   1. Navigate to target workspace")
    print(f"   2. Go to {target_parent_path}")
    print("   3. Verify dashboards load correctly")
    print("   4. Test data with new catalog/schema")
    print("   5. Verify permissions applied")
    
else:
    print("\n❌ Bundle deployment failed")
    
    if deploy_result.get('stderr'):
        print(f"\nError output:\n{deploy_result['stderr']}")
    if deploy_result.get('error'):
        print(f"\nError: {deploy_result['error']}")
    
    print("\n💡 Troubleshooting:")
    print("   - Check target workspace credentials")
    print("   - Verify warehouse exists and is accessible")
    print("   - Check parent folder permissions")
    print("   - Review bundle validation output above")
    
    raise Exception("Bundle deployment failed")

## Cell 7: Verify Deployment

In [ ]:
print("\n" + "="*80)
print("VERIFYING DEPLOYMENT")
print("="*80)

print("\n🔍 Connecting to target workspace...")
target_client = create_workspace_client('target')

print("\n📋 Listing deployed dashboards...")
deployed_dashboards = []

for dash in target_client.lakeview.list():
    if dash.parent_path and dash.parent_path.startswith(target_parent_path):
        deployed_dashboards.append({
            'Name': dash.display_name,
            'ID': dash.dashboard_id,
            'Path': dash.parent_path
        })

if deployed_dashboards:
    print(f"\n✅ Found {len(deployed_dashboards)} deployed dashboard(s)\n")
    df = pd.DataFrame(deployed_dashboards)
    display(df)
    
    print("\n🎉 Migration complete!")
    print("\n📝 Verification checklist:")
    print("   □ Dashboards visible in target workspace")
    print("   □ Data loads correctly (new catalog/schema)")
    print("   □ Visualizations render properly")
    print("   □ Permissions applied correctly")
    print("   □ Warehouse connection works")
else:
    print("\n⚠️  No dashboards found at expected location")
    print(f"   Expected path: {target_parent_path}")
    print("\n💡 Check deployment logs above for errors")